> **Notebook d'approfondissement** — mémoire, vitesse, types catégoriels, lecture optimisée, `.eval()`.

# Performance et types catégoriels

In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
import numpy as np
from _data import load_accidents, load_penguins

df = load_accidents()
df.shape

## 1. Mesurer la mémoire d'un DataFrame

Avant d'optimiser, il faut mesurer.

In [ ]:
# Mémoire par colonne (en octets)
df.memory_usage(deep=True).sort_values(ascending=False)

In [ ]:
# Total en Mo
total_mo = df.memory_usage(deep=True).sum() / 1024**2
print(f"Mémoire totale : {total_mo:.1f} Mo")

## 2. Le type `object` : la colonne la plus gourmande

Une colonne de type `object` (chaîne de caractères Python) stocke des **pointeurs** vers des objets
Python alloués séparément dans la mémoire. Chaque valeur prend 50–200 octets.

C'est la source numéro un de consommation mémoire excessive en pandas.

In [ ]:
# Colonnes object dans notre dataset
df.select_dtypes("object").columns.tolist()

In [ ]:
# Cardinalité : combien de valeurs uniques dans chaque colonne object ?
# Une faible cardinalité = candidat idéal pour Categorical
for col in df.select_dtypes("object").columns:
    n_unique = df[col].nunique()
    n_total  = len(df[col].dropna())
    print(f"{col:12s} : {n_unique:5d} valeurs uniques / {n_total} lignes  ({n_unique/n_total*100:.1f}%)")

## 3. Le dtype `Categorical` — principe et mémoire

Au lieu de stocker `"75"`, `"75"`, `"69"`, `"75"` en mémoire 4 fois,
pandas stocke un **dictionnaire** `{0: "69", 1: "75"}` et un tableau d'entiers `[1, 1, 0, 1]`.

L'économie est massive quand la cardinalité est faible (peu de valeurs uniques, beaucoup de lignes).

In [ ]:
# Colonne dep : object vs Categorical
dep_object = df["dep"]  # type object
dep_cat    = df["dep"].astype("category")

mem_obj = dep_object.memory_usage(deep=True)
mem_cat = dep_cat.memory_usage(deep=True)

print(f"object      : {mem_obj:8,} octets")
print(f"category    : {mem_cat:8,} octets")
print(f"Ratio       : {mem_obj / mem_cat:.1f}x plus léger")

In [ ]:
# Convertir toutes les colonnes à faible cardinalité en Categorical
SEUIL_CARDINALITE = 0.05  # moins de 5% de valeurs uniques

df_opt = df.copy()
for col in df_opt.select_dtypes("object").columns:
    ratio = df_opt[col].nunique() / len(df_opt[col].dropna())
    if ratio < SEUIL_CARDINALITE:
        df_opt[col] = df_opt[col].astype("category")

avant = df.memory_usage(deep=True).sum() / 1024**2
apres = df_opt.memory_usage(deep=True).sum() / 1024**2
print(f"Avant : {avant:.1f} Mo")
print(f"Après : {apres:.1f} Mo")
print(f"Gain  : {avant - apres:.1f} Mo ({(1 - apres/avant)*100:.0f}%)")

## 4. Categorical ordonné — pour les variables ordinales

Un `Categorical` peut être **ordonné** pour représenter des variables ordinales
(petit < moyen < grand). Ça active les comparaisons `<`, `>` et les tris respectent l'ordre logique.

In [ ]:
# Gravité des accidents (1=indemne, 2=tué, 3=blessé hosp., 4=blessé léger)
# Pour l'exemple, on crée un Categorical ordonné sur la colonne lum
# lum : 1=jour, 2=crépuscule, 3=nuit éclairée, 4=nuit non éclairée, 5=nuit sans écl.

lum_ordonne = pd.CategoricalDtype(
    categories=[1, 2, 3, 4, 5],
    ordered=True,
)
df["lum_cat"] = df["lum"].astype(lum_ordonne)

# Comparaisons ordinales fonctionnent
(df["lum_cat"] >= 3).sum()  # nombre d'accidents de nuit (lum >= 3)

In [ ]:
# Le tri respecte l'ordre défini, pas l'ordre alphabétique
df[["lum", "lum_cat"]].drop_duplicates().sort_values("lum_cat")

## 5. Performance de `groupby` avec Categorical

pandas peut être plus rapide sur les groupby de colonnes catégorielles
car il connaît à l'avance la liste des groupes possibles.

In [ ]:
dep_object = df["dep"]
dep_cat    = df["dep"].astype("category")

In [ ]:
%%timeit
df.groupby(dep_object)["Num_Acc"].count()

In [ ]:
%%timeit
df.groupby(dep_cat)["Num_Acc"].count()

## 6. Lecture optimisée avec `read_csv()`

Deux paramètres font une grande différence à la lecture :

In [ ]:
from _data import ACCIDENTS_CARACTERISTIQUES_URL

# Lire seulement les colonnes utiles — évite de charger 15 colonnes si on en utilise 4
df_lean = pd.read_csv(
    ACCIDENTS_CARACTERISTIQUES_URL,
    sep=";",
    usecols=["Num_Acc", "dep", "mois", "lum"],   # uniquement ces colonnes
    dtype={"dep": "category"},                    # parser dep directement en Categorical
)

mem_lean = df_lean.memory_usage(deep=True).sum() / 1024**2
mem_full = df.memory_usage(deep=True).sum() / 1024**2
print(f"Toutes colonnes : {mem_full:.1f} Mo")
print(f"4 colonnes     : {mem_lean:.1f} Mo")

## 7. Lecture par chunks — traiter un fichier trop grand pour la RAM

In [ ]:
# chunksize : lire le fichier bloc par bloc sans tout charger
# Utile pour des agrégations sur des fichiers de plusieurs Go

totaux_par_dep = pd.Series(dtype=int)

reader = pd.read_csv(
    ACCIDENTS_CARACTERISTIQUES_URL,
    sep=";",
    usecols=["Num_Acc", "dep"],
    chunksize=10_000,
)

for chunk in reader:
    comptage = chunk.groupby("dep")["Num_Acc"].count()
    totaux_par_dep = totaux_par_dep.add(comptage, fill_value=0)

totaux_par_dep.nlargest(10)

## 8. Parquet vs CSV — vitesse et taille

Parquet est un format binaire colonaire. Il est compressé, typé, et beaucoup plus rapide à lire
que CSV, surtout quand on ne lit que quelques colonnes (`usecols` est gratuit en Parquet).

In [ ]:
import os

# Écrire en CSV et en Parquet
df.to_csv("accidents_bench.csv", index=False)
df.to_parquet("accidents_bench.parquet")

taille_csv     = os.path.getsize("accidents_bench.csv") / 1024**2
taille_parquet = os.path.getsize("accidents_bench.parquet") / 1024**2
print(f"CSV     : {taille_csv:.1f} Mo")
print(f"Parquet : {taille_parquet:.1f} Mo")
print(f"Ratio   : {taille_csv / taille_parquet:.1f}x plus léger en Parquet")

In [ ]:
%%timeit
pd.read_csv("accidents_bench.csv")

In [ ]:
%%timeit
pd.read_parquet("accidents_bench.parquet")

In [ ]:
%%timeit
# Parquet : lire seulement 2 colonnes — le reste n'est même pas décompressé
pd.read_parquet("accidents_bench.parquet", columns=["dep", "mois"])

In [ ]:
# Nettoyage
os.remove("accidents_bench.csv")
os.remove("accidents_bench.parquet")

## 9. `.eval()` — expressions vectorisées rapides

`df.eval()` évalue une expression string dans le contexte du DataFrame.
Avec le moteur `numexpr`, il évite la création d'objets Python intermédiaires
et peut utiliser plusieurs cœurs CPU.

In [ ]:
# Benchmark : pandas normal vs .eval()
df_num = df[["vma", "nbv", "lartpc"]].dropna()

In [ ]:
%%timeit
df_num["vma"] * df_num["nbv"] + df_num["lartpc"]

In [ ]:
%%timeit
df_num.eval("vma * nbv + lartpc")

## 10. Identifier les goulots : `iterrows` et `apply`

Les deux opérations les plus lentes en pandas sont :
1. `iterrows()` / `itertuples()` — boucle Python sur les lignes
2. `apply()` avec une fonction Python non-vectorisée

L'alternative est presque toujours une opération vectorisée.

In [ ]:
sample = df[["mois", "lum"]].head(5_000)

def classer_periode(row):
    if row["lum"] <= 1:
        return "jour"
    return "nuit"

In [ ]:
%%timeit
# Boucle iterrows — 100x plus lent
[classer_periode(row) for _, row in sample.iterrows()]

In [ ]:
%%timeit
# apply — mieux qu'iterrows, toujours lent
sample.apply(classer_periode, axis=1)

In [ ]:
%%timeit
# Vectorisé avec np.where — imbattable
np.where(sample["lum"] <= 1, "jour", "nuit")

## Récapitulatif : règles d'or pour la performance pandas

1. **Mesurer avant d'optimiser** : `memory_usage(deep=True)` et `%%timeit`.
2. **Colonnes catégorielles** : convertir les `object` à faible cardinalité en `category` — gain mémoire 5-20x, groupby plus rapide.
3. **Lecture optimisée** : `usecols` + `dtype` dans `read_csv()` — charger uniquement ce qu'on utilise.
4. **Format Parquet** : 3-10x plus léger que CSV, 5-20x plus rapide à lire, lecture de colonnes sélectives gratuite.
5. **Éviter `iterrows`** : toujours chercher une opération vectorisée (`np.where`, `.map()`, `.str.`, `.dt.`).
6. **`apply()` en dernier recours** : si obligatoire, préférer `axis=0` (colonnes) à `axis=1` (lignes).
7. **Fichiers trop grands** : `chunksize` pour traiter par blocs, ou passer à DuckDB / polars lazy.